# 🎛️ Aula 15 — Gestão de Processos e Carga de Trabalho

**Objetivo:** gerenciar execuções **concorrentes** em GPU com Bash — **exclusão mútua**
(`flock`/lock), **filas com prioridade**, **monitoramento de processos** em tempo real e
**agendamento** — garantindo uso justo e eficiente da GPU em ambientes multiusuário, sem
depender de Slurm ou Kubernetes.

**Roteiro deste notebook:**
1. Verificação do ambiente.
2. Teoria: race condition, mutex e fila de prioridade.
3. Demo: lock por diretório (exclusão mútua) simulado em Python.
4. Demo: fila por prioridade (`prioridade_timestamp_nome`).
5. Monitoramento de processos na GPU.
6. Agendamento (cron/systemd) e prioridade (`nice`/`ionice`).
7. Discussão, **Exercícios (5)** e síntese.

> ℹ️ No **Colab** não há `flock`, `systemd` nem `nvidia-smi` de verdade (runtime efêmero).
> Por isso os conceitos são **simulados em Python** com o mesmo comportamento. No
> **laboratório Windows** (`laboratorio_windows/`), os scripts Bash rodam de verdade.

## 1. Verificação do Ambiente

In [ ]:
# @title 🔍 Ambiente: Python, SO e presença de GPU
# ============================================================================
# OBJETIVO: saber o que temos. No Colab não há flock/systemd; aqui usamos Python
# para SIMULAR o comportamento (lock e fila).
# ============================================================================
import os, platform, shutil

print(f"Sistema : {platform.system()} {platform.release()}")
print(f"Python  : {platform.python_version()}")
print(f"Núcleos : {os.cpu_count()}")
print(f"nvidia-smi : {'sim' if shutil.which('nvidia-smi') else 'nao (modo simulado)'}")
print("flock (Linux) :", "sim" if shutil.which('flock') else "nao (usaremos lock por diretorio em Python)")

## 2. Teoria: race condition, mutex e fila

Quando **vários jobs** usam a **mesma GPU**, eles competem por VRAM e capacidade de
processamento → **CUDA OOM**, *crashes* silenciosos e resultados corrompidos.

```
[08:00:00] Job-A inicia — aloca 6 GB VRAM
[08:00:01] Job-B inicia — aloca 6 GB VRAM
[08:00:02] Job-A: CUDA OOM — apenas 2 GB livres!
[08:00:02] Job-B: crash silencioso
[08:00:10] Com o lock: jobs executam 1 por vez, sem conflito
```

A correção tem duas partes:
1. **Exclusão mútua** (mutex): garantir **um job por vez** — no Linux, `flock -x`;
   portável, **lock por diretório** (`mkdir` é atômico).
2. **Fila com prioridade:** em vez de corrida livre, os jobs entram numa fila e executam
   na ordem correta.

> **`nice`/`ionice`** ajustam a prioridade de CPU/IO de um processo; a **fila** ajusta a
> prioridade de *entrada* na GPU.

## 3. Demo: exclusão mútua (lock por diretório)

O lock é um **diretório**. Criar diretório com `mkdir` é **atômico**: se dois processos
tentam ao mesmo tempo, só um consegue. É o mesmo princípio do `flock -x`, mas funciona
também no Windows. Aqui simulamos isso com o sistema de arquivos real do Colab.

In [ ]:
# @title 🔒 Lock por diretório — exclusão mútua
# ============================================================================
# OBJETIVO: garantir que apenas UM 'job' use a GPU por vez.
# Usamos os.mkdir, que e atomico: se o diretorio ja existe, falha.
# ============================================================================
import os, time, shutil

DIR_LOCK = "lock_gpu"
shutil.rmtree(DIR_LOCK, ignore_errors=True)   # limpa estado anterior

def tentar_lock():
    """Tenta adquirir o lock. Devolve True se conseguiu, False se ja estava ocupado."""
    try:
        os.mkdir(DIR_LOCK)      # atomico: so um consegue
        return True
    except FileExistsError:
        return False

def liberar_lock():
    shutil.rmtree(DIR_LOCK, ignore_errors=True)

# Simula dois jobs querendo a GPU. O segundo so entra apos o primeiro liberar.
print("Job-A tentando o lock...", tentar_lock())
print("Job-B tentando o lock...", tentar_lock(), "(esperado: False -> GPU ocupada)")
liberar_lock()
print("Job-A liberou. Job-B tentando de novo...", tentar_lock())
liberar_lock()
print("\nConclusao: sem o lock, os dois rodariam juntos -> OOM.")

## 4. Demo: fila por prioridade

Cada job cria um **ticket** com o nome `prioridade_timestamp_nome`. Como `sort` é
lexicográfico, **`1_...` vem antes de `2_...` antes de `3_...`**; dentro da mesma
prioridade, o *timestamp* desempata (FIFO).

In [ ]:
# @title 🎫 Fila por prioridade (menor numero = mais alta)
# ============================================================================
# OBJETIVO: ordenar jobs por prioridade e, dentro dela, por ordem de chegada.
# ============================================================================
import time

def ticket(prioridade, nome):
    """Monta o nome do ticket: prioridade_timestamp_nome."""
    return f"{prioridade}_{int(time.time()*1000):013d}_{nome}"

# 4 jobs chegam com prioridades e tempos diferentes (simulados com milissegundos).
base = int(time.time()*1000)
jobs = [
    (1, "Job-Alta-A"),
    (3, "Job-Baixa-B"),
    (2, "Job-Media-C"),
    (1, "Job-Alta-D"),
]
fila = [f"{pr}_{base + i:013d}_{nome}" for i, (pr, nome) in enumerate(jobs)]

print("Ordem de chegada:")
for t in fila:
    print("  ", t)

print("\nOrdem de EXECUCAO (ordenada por prioridade, depois timestamp):")
for i, t in enumerate(sorted(fila), 1):
    print(f"  {i}. {t}")

print("\nEsperado: Alta-A -> Alta-D -> Media-C -> Baixa-B")

## 5. Monitoramento de processos na GPU

No Linux, `nvidia-smi pmon` acompanha **quais processos** usam a GPU e quanta VRAM cada
um consome. No laboratório Windows, usamos os **contadores de desempenho** do Windows.

Combinando com o lock, dá para responder: *quem está segurando a GPU agora?*

In [ ]:
# @title 📊 Quem está usando a GPU? (com o lock como fonte)
# ============================================================================
# OBJETIVO: mostrar o 'dono' atual do lock (equivalente ao pmon no Windows).
# ============================================================================
import os, shutil

DIR_LOCK = "lock_gpu"

def dono_do_lock():
    if os.path.isdir(DIR_LOCK):
        # Se houvesse um arquivo 'pid' dentro, leriamos o PID do dono.
        arquivo_pid = os.path.join(DIR_LOCK, "pid")
        if os.path.exists(arquivo_pid):
            with open(arquivo_pid) as f:
                return f.read().strip()
        return "(ocupado, dono desconhecido)"
    return None

# Simula um job pegando a GPU: cria o lock e grava o 'pid'
shutil.rmtree(DIR_LOCK, ignore_errors=True)
os.mkdir(DIR_LOCK)
with open(os.path.join(DIR_LOCK, "pid"), "w") as f:
    f.write(str(os.getpid()))

print(f"GPU ocupada por PID: {dono_do_lock()}")
shutil.rmtree(DIR_LOCK, ignore_errors=True)
print(f"Apos liberar: {dono_do_lock()}")
print("\nNo Linux, o equivalente e: nvidia-smi pmon -s u")

## 6. Agendamento e prioridade de processo

### cron × systemd

| | **cron** | **systemd** |
| :--- | :--- | :--- |
| Tipo | periódico (agenda) | contínuo (serviço) |
| Reinicia sozinho | não | **sim** |
| Inicia no boot | não | **sim** |
| Melhor para | relatórios, limpeza | jobs de longa duração |

### Prioridade de processo (Linux)

```bash
nice -n 19 python3 train.py      # menor prioridade de CPU
ionice -c 3 python3 train.py     # idle I/O
renice -n -5 -p PID              # aumentar prioridade de um job em curso
```

> **Starvation:** jobs de baixa prioridade podem nunca executar se os de alta ocuparem a GPU
> continuamente. O remédio clássico é o **aging** (aumentar a prioridade com o tempo de espera).

In [ ]:
# @title ⏰ Referência de agendamento (cron e systemd)
# ============================================================================
# OBJETIVO: ver a sintaxe de um cron job e de um unit systemd (nao executamos
# no Colab, que nao tem init).
# ============================================================================
print("Crontab — rodar o job todo dia as 08:00:")
print("  0 8 * * * /caminho/fila_gpu.sh 1 JobDiario 'python train_job.py'")
print()
print("Unit systemd (gpu-job@.service) — isolamento e restart:")
print("  [Service]")
print("  ExecStart=/caminho/fila_gpu.sh %i")
print("  MemoryMax=8G")
print("  Restart=on-failure")
print()
print("No laboratorio Windows, o agendar.sh simula isso e mostra o schtasks.")

## 7. Discussão em Grupo

Em grupos de 3–4, no cenário de **1 GPU e 4 alunos**:

1. Por que executar jobs em paralelo na mesma GPU pode **corromper resultados**?
2. Como a **fila por prioridade** garante uso justo? O que é *starvation* e como o **aging**
   resolve?
3. Qual a diferença entre usar **`flock`** (laboratório) e **systemd** (produção)?
4. Como integrar o monitor de processos da Aula 14 a esta solução de fila?

## 8. Exercícios (5)

Resolva os 5 exercícios **neste notebook**. O valor está em **experimentar e explicar**.

---

**1) Race condition.** Explique, com suas palavras, o que é uma *race condition* quando dois
jobs disputam a GPU. Que erro concreto aparece?

**2) Mutex.** Por que o **lock por diretório** (`mkdir`) funciona como exclusão mútua? O que
torna a operação `mkdir` especial em relação a criar um arquivo?

**3) Fila por prioridade.** Na célula-esqueleto, monte os tickets de uma lista de jobs e
imprima a ordem de execução. Confirme `Alta-A → Alta-D → Media-C → Baixa-B`.

**4) Starvation e aging.** Dê um exemplo em que um job de prioridade 3 **nunca** executa. Como
o *aging* (subir a prioridade com o tempo de espera) corrige isso?

**5) flock × systemd.** Para cada caso, escolha a ferramenta e justifique: *(a)* um script
ad-hoc de teste no laboratório; *(b)* um serviço de treino que roda todo dia e precisa de
logging e restart automático.

---

> 💡 **No laboratório Windows**, coloque tudo isso em prática com
> `laboratorio_windows/teste_fila.sh` e `monitor_gpu_proc.sh`.

In [ ]:
# @title Exercício 3 — complete a ordenação da fila
# ============================================================================
# OBJETIVO: montar os tickets e imprimir a ordem de execucao.
# Complete os TODOs.
# ============================================================================
import time

def ticket(prioridade, nome, ts):
    return f"{prioridade}_{ts:013d}_{nome}"

base = int(time.time() * 1000)
jobs = [(1, "Job-Alta-A"), (3, "Job-Baixa-B"), (2, "Job-Media-C"), (1, "Job-Alta-D")]

fila = []
for i, (pr, nome) in enumerate(jobs):
    # TODO: adicione o ticket (prioridade, nome, base + i) a 'fila'
    pass

print("Ordem de execucao:")
# TODO: ordene a fila (sorted) e imprima numerada

# Esperado: 1. Alta-A  2. Alta-D  3. Media-C  4. Baixa-B

## 9. Síntese e Tarefa de Casa

**O que levar:**
- **Race condition:** jobs concorrentes → OOM e resultados corrompidos.
- **Mutex:** `flock -x` (Linux) ou **lock por diretório** (`mkdir` atômico).
- **Fila:** ticket `prioridade_timestamp_nome` → `sort` decide a ordem (prioridade + FIFO).
- **Monitoramento:** `nvidia-smi pmon` (Linux) / contadores do Windows.
- **Prioridade:** `nice`/`ionice`; **systemd** para produção (restart, logging, cgroups).
- **Starvation/aging:** justiça na fila ao longo do tempo.

**Tarefa (opcional):** no **laboratório Windows** (`laboratorio_windows/`):
1. rode `teste_fila.sh` e observe a ordem por prioridade;
2. adicione um 5º job de **prioridade 1** após 30 s e confirme que ele "fura" a fila;
3. implemente **aging** (jobs esperando > 10 min sobem de prioridade);
4. integre `monitor_gpu_proc.sh` ao pipeline da Aula 14.

> 🔗 **Fim do Bloco 3.** O código paralelo (Aulas 7–13), o monitoramento contínuo (Aula 14) e
> a gestão de concorrência (Aula 15) formam a operação completa de uma GPU em produção.